<a href="https://colab.research.google.com/github/dhanushkumar2968/Car-colour-detection-Model/blob/main/Car_colour_detection_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Import Libraries
import cv2
import numpy as np
from ultralytics import YOLO
from tkinter import filedialog
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files

ModuleNotFoundError: No module named 'ultralytics'

In [ ]:
#Load YOLO Model
model = YOLO("yolov8n.pt")

In [ ]:
#Upload Image
uploaded=files.upload()
file_path = list(uploaded.keys())[0]

image = cv2.imread(file_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

In [ ]:
#Detect Objects
results = model(image_rgb)

people_count = 0

In [ ]:
#Color Detection Function
def detect_color(car_crop):

    avg_color = np.mean(car_crop, axis=(0,1))

    blue = avg_color[2]
    green = avg_color[1]
    red = avg_color[0]

    if blue > red and blue > green:
        return "Blue"

    return "Other"

In [ ]:
#Draw Boxes
for result in results:

    boxes = result.boxes

    for box in boxes:

        cls = int(box.cls[0])

        x1, y1, x2, y2 = map(int, box.xyxy[0])

        # Car Class
        if cls == 2:

            car_crop = image_rgb[y1:y2, x1:x2]

            color_name = detect_color(car_crop)

            # Red box for blue cars
            if color_name == "Blue":
                color = (255, 0, 0)
            else:
                color = (0, 0, 255)

            cv2.rectangle(image_rgb, (x1,y1), (x2,y2), color, 3)

            cv2.putText(
                image_rgb,
                color_name,
                (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                color,
                2
            )

        # Person Class
        elif cls == 0:

            people_count += 1

            cv2.rectangle(image_rgb, (x1,y1), (x2,y2), (0,255,0), 2)

In [ ]:
#Show People Count
cv2.putText(
    image_rgb,
    f"People Count: {people_count}",
    (20,40),
    cv2.FONT_HERSHEY_SIMPLEX,
    1,
    (255,255,0),
    3
)

In [ ]:
#Display Result
plt.figure(figsize=(14,8))
plt.imshow(image_rgb)
plt.axis("off")
plt.show()